In [ ]:
import transformers
import torch
import datetime
import time
import json
import re
import numpy as np


In [ ]:
test_data_path = "" # test data path
model_id = "" # model path
torch.cuda.empty_cache()
 
pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.float16},
    device= torch.device("cuda:0" if torch.cuda.is_available() else "cpu"),
)
 

In [10]:
fewshot_prompt = open("").read() # few-shot prompt path

In [12]:
ANS_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
INVALID_ANS = "[invalid]"
def extract_answer_hf(completion):
    match = ANS_RE.search(completion)
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return eval(match_str)
    else:
        return INVALID_ANS


def extract_answer(completion):
    try:
        last_number = re.findall(r"\d+", completion)[-1]
        return eval(last_number)
    except:
        return INVALID_ANS


def is_correct(completion, answer):
    gold = extract_answer_hf(answer)
    print("Answer: "+ str(gold) + "-------" + "Completion: "+ str(extract_answer(completion)))
    assert gold != INVALID_ANS, "No ground truth answer found in the document."
    return extract_answer(completion) == gold

In [ ]:
acc_res = []
acc_answer = []
total_start_time = time.time()
with open(test_data_path, 'r', encoding='utf-8') as f1:
    json_content = json.load(f1)
    for i in json_content:
        each_start_time = time.time()
        prompt = fewshot_prompt +i['query']+ "\n"
        terminators = [
            pipeline.tokenizer.eos_token_id,
            pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]
        outputs = pipeline(
            prompt,
            max_new_tokens=512,
            eos_token_id=terminators,
            do_sample=False,
            temperature=0.1,
            top_p=0.9,
        )
        completion=outputs[0]["generated_text"][len(prompt):]
        completion = completion.split("Question:")[0]
        answer = i["response"]
        acc = is_correct(completion, answer)
        acc_answer.append(acc)  
        i["completion"] = completion
        i["acc"] = acc
        each_end_time = time.time()
        elapsed_time = each_end_time - each_start_time
        current_time = datetime.datetime.now()
        text_to_write = "in case:"+ str(cnt) + "!" +" Current time: " + str(current_time) + " , Elapsed Time: " + str(elapsed_time) + " seconds"  
        print(text_to_write)
        acc_res.append(completion)

total_end_time = time.time()
total_time = total_end_time - total_start_time
print("Acc: ", np.mean(acc_answer))
print("Total time: "+ str(total_time) + " seconds!")   

file_path = "" 
with open(file_path, "a") as file:  
    file.write(text_to_write)